### 🛠️ Environment Setup & Dependency Verification
This notebook includes verified dependencies to ensure reproducible execution.

- **Automatic Setup:** Cell 2 verifies Python version compatibility and installs verified package versions sequentially.
- **Network Notice:** Active internet access is required to download uncached packages.


In [ ]:
# =====================================================================
# VERIFIED ENVIRONMENT DEPENDENCIES (2026-08-29 17:22:58)
# =====================================================================

import sys
import subprocess
import importlib.metadata

REQUIRED_PYTHON = (3, 12)
CURRENT_PYTHON = (sys.version_info.major, sys.version_info.minor)

# Major version mismatch -> Clean hard stop
if CURRENT_PYTHON[0] != REQUIRED_PYTHON[0]:
    req_major = REQUIRED_PYTHON[0]
    curr_major = CURRENT_PYTHON[0]
    print(f"❌ Error: Major Python version mismatch!")
    print(f"This notebook requires Python {req_major}.x, but your environment is running Python {curr_major}.x.\n")
    sys.exit("Execution stopped due to Python major version incompatibility.")

# Minor version mismatch -> Non-blocking warning
if CURRENT_PYTHON[1] != REQUIRED_PYTHON[1]:
    req_ver = f"{REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}"
    curr_ver = f"{CURRENT_PYTHON[0]}.{CURRENT_PYTHON[1]}"
    print(f"⚠️ This code was created with Python {req_ver}. You are trying to run it with {curr_ver}.")
    print(f"If installation fails, consider changing your runtime Python version back to {req_ver}.\n")

# Dependency Specification with Scoped Flags
DEPENDENCIES = [{'name': 'numpy', 'version': '2.0.2', 'flags': []}, {'name': 'pandas', 'version': '2.2.3', 'flags': []}]

print(f"Applying verified environment dependencies [2026-08-29 17:22:58]...")
print("💡 Note: Dependencies are installed sequentially to prevent index conflicts.\n")

passed_count = 0
failed_packages = []
total_deps = len(DEPENDENCIES)
installed_baseline = {}

for idx, item in enumerate(DEPENDENCIES, start=1):
    name = item["name"]
    ver = item.get("version", "")
    flags = item.get("flags", [])
    specifier = f"{name}=={ver}" if ver else name

    # Step 1: Pre-install inspection
    # Avoids redundant re-installations in pre-configured platforms (Colab, Kaggle)
    already_satisfied = False
    try:
        current_ver = importlib.metadata.version(name)
        if not ver or current_ver == ver:
            already_satisfied = True
            passed_count += 1
            installed_baseline[name] = current_ver
            print(f"[{idx}/{total_deps}] ⚡ {name} ({current_ver}) already satisfied in environment")
    except Exception:
        pass

    if already_satisfied:
        continue

    # Step 2: Non-blocking, progressive streaming installation
    # Flags explanation:
    # - "--no-input": Prevents pip from prompting for credentials or confirmation on stdin
    # - "--disable-pip-version-check": Eliminates network overhead checking for newer pip releases
    # - "--no-warn-script-location": Suppresses path warnings for binaries installed into user/local bins
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-input",
        "--disable-pip-version-check",
        "--no-warn-script-location",
        specifier
    ] + flags

    print(f"[{idx}/{total_deps}] 📦 Installing {specifier}...")
    sys.stdout.flush()

    # Streaming subprocess configuration:
    # - stdin=subprocess.DEVNULL: Closes stdin to guarantee subprocess cannot block waiting on interactive input
    # - stdout=subprocess.PIPE, stderr=subprocess.STDOUT: Merges output streams to preserve chronological output
    # - text=True, bufsize=1: Enables line-buffered text mode for real-time progress logging
    # - timeout=180: Safeguards against dead socket stalls by raising TimeoutExpired after 3 minutes per package
    captured_output = []
    try:
        proc = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        stdout_data, _ = proc.communicate(timeout=120)
        returncode = proc.returncode
        for line in stdout_data.splitlines():
            if line.strip():
                print(f"    {line}")
        sys.stdout.flush()
    except subprocess.TimeoutExpired:
        proc.kill()
        stdout_data, _ = proc.communicate()
        returncode = -1
        print("    ❌ Installation timed out after 120s.")

    except subprocess.TimeoutExpired:
        if proc:
            proc.kill()
        returncode = -1
        captured_output.append("Error: Subprocess installation exceeded per-package timeout limit (180s).")
        print("    ❌ Installation timed out after 180s.")
        sys.stdout.flush()

    except Exception as exc:
        returncode = -1
        captured_output.append(f"Execution failed: {exc}")
        print(f"    ❌ Execution failed: {exc}")
        sys.stdout.flush()

    if returncode == 0:
        passed_count += 1
        print(f"    ✅ {specifier} installed successfully")
        
        # Real-time drift audit across previously installed dependencies
        try:
            current_ver = importlib.metadata.version(name)
            installed_baseline[name] = current_ver
        except Exception:
            pass

        for prev_pkg, prev_ver in list(installed_baseline.items()):
            if prev_pkg == name:
                continue
            try:
                active_now = importlib.metadata.version(prev_pkg)
                if active_now != prev_ver:
                    print(f"   ⚠️ Dependency Drift: Installing '{specifier}' caused '{prev_pkg}' to drift from {prev_ver} ➔ {active_now}")
                    installed_baseline[prev_pkg] = active_now
            except Exception:
                pass
    else:
        err_snippet = captured_output[-1] if captured_output else "Unknown pip error"
        failed_packages.append((specifier, ver, flags, "\n".join(captured_output)))
        print(f"    ❌ {specifier} failed to install (exit code {returncode})")
        print(f"       ├─ Author Verified Version: {ver or 'unspecified'}")
        if flags:
            print(f"       ├─ Scoped Flags: {' '.join(flags)}")
        print(f"       └─ Error: {err_snippet}\n")

print("\n" + "=" * 60)
if not failed_packages:
    print(f"✅ Setup complete! All {passed_count}/{total_deps} dependencies verified.")
else:
    print(f"⚠️ Setup completed with issues: {passed_count}/{total_deps} packages installed.")
    print("Troubleshooting Steps:")
    print("1. Internet Access: Ensure your notebook environment has active internet access.")
    print("2. Unpinned Installs: Test installing failed libraries manually: '!pip install <pkg>'")
    print(f"3. Troubleshooting Steps: For a detailed guide on resolving setup errors, see: https://github.com/flyinacres/notebook_env/blob/main/HELP.md")
print("=" * 60)


In [ ]:
# Cell 1: Standard pre-installed imports
import numpy as np
import pandas as pd

In [ ]:
# Cell 2: Execution and functional verification
arr = np.array([1, 2, 3])
df = pd.DataFrame({'a': arr})
assert df['a'].sum() == 6, 'Pandas dataframe calculation failed'
print('Canary check passed: numpy and pandas operational')